 Joins y Uniones en Spark DataFrames

In [2]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._

val spark = SparkSession.builder()
  .appName("Joins-Ejemplos")
  .master("local[*]")
  .config("spark.sql.shuffle.partitions", "4")
  .config("spark.sql.crossJoin.enabled", "true")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
import spark.implicits._

println(s"Spark ${spark.version} — Scala ${scala.util.Properties.versionString} ✅")

Spark 4.1.1 — Scala version 2.13.18 ✅


import $ivy.$
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@306edfea
import spark.implicits._

In [3]:
// DataFrame A — clientes
val clientes = Seq(
  (1, "Ana"),
  (2, "Borja"),
  (3, "Carmen")
).toDF("id", "nombre")

// DataFrame B — pedidos
val pedidos = Seq(
  ("P1", 1, 200.0),
  ("P2", 1,  80.0),
  ("P3", 2, 350.0)
).toDF("ped", "clienteId", "importe")

println("=== DataFrame A: clientes ===")
clientes.show()

println("=== DataFrame B: pedidos ===")
pedidos.show()

=== DataFrame A: clientes ===
+---+------+
| id|nombre|
+---+------+
|  1|   Ana|
|  2| Borja|
|  3|Carmen|
+---+------+

=== DataFrame B: pedidos ===
+---+---------+-------+
|ped|clienteId|importe|
+---+---------+-------+
| P1|        1|  200.0|
| P2|        1|   80.0|
| P3|        2|  350.0|
+---+---------+-------+



clientes: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string]
pedidos: org.apache.spark.sql.package.DataFrame = [ped: string, clienteId: int ... 1 more field]

¿Qué clientes han realizado al menos un pedido?

Técnica: INNER JOIN — devuelve únicamente las filas que tienen coincidencia en ambos DataFrames.

In [4]:

// INNER JOIN
val inner = clientes.join(pedidos, clientes("id") === pedidos("clienteId"), "inner")

println("=== INNER JOIN ===")
inner.show()
println(s"Filas: ${inner.count()}")

=== INNER JOIN ===
+---+------+---+---------+-------+
| id|nombre|ped|clienteId|importe|
+---+------+---+---------+-------+
|  1|   Ana| P1|        1|  200.0|
|  1|   Ana| P2|        1|   80.0|
|  2| Borja| P3|        2|  350.0|
+---+------+---+---------+-------+

Filas: 3


inner: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 3 more fields]

¿Cuáles son todos nuestros clientes y sus pedidos, incluyendo los que aún no han comprado?

Técnica: LEFT JOIN — devuelve todas las filas del DataFrame izquierdo (clientes) y las coincidencias 

del derecho (pedidos). Donde no hay coincidencia, rellena con null.

In [5]:
// LEFT JOIN
val left = clientes.join(pedidos, clientes("id") === pedidos("clienteId"), "left")

println("=== LEFT JOIN ===")
left.show()
println(s"Filas: ${left.count()}")

=== LEFT JOIN ===
+---+------+----+---------+-------+
| id|nombre| ped|clienteId|importe|
+---+------+----+---------+-------+
|  1|   Ana|  P2|        1|   80.0|
|  1|   Ana|  P1|        1|  200.0|
|  2| Borja|  P3|        2|  350.0|
|  3|Carmen|NULL|     NULL|   NULL|
+---+------+----+---------+-------+

Filas: 4


left: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 3 more fields]

¿Existen pedidos cuyo cliente no está registrado en nuestro sistema?

Técnica: RIGHT JOIN — devuelve todas las filas del DataFrame derecho (pedidos) y las coincidencias 

del izquierdo (clientes). Donde no hay coincidencia, rellena con null.

In [6]:
// RIGHT JOIN
val right = clientes.join(pedidos, clientes("id") === pedidos("clienteId"), "right")

println("=== RIGHT JOIN ===")
right.show()
println(s"Filas: ${right.count()}")

=== RIGHT JOIN ===
+---+------+---+---------+-------+
| id|nombre|ped|clienteId|importe|
+---+------+---+---------+-------+
|  1|   Ana| P1|        1|  200.0|
|  1|   Ana| P2|        1|   80.0|
|  2| Borja| P3|        2|  350.0|
+---+------+---+---------+-------+

Filas: 3


right: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 3 more fields]

¿Hay clientes sin pedidos O pedidos sin cliente registrado? Necesitamos detectar ambos casos a la vez.

Técnica: FULL OUTER JOIN — devuelve todas las filas de ambos DataFrames. Donde no hay coincidencia en 

ninguno de los dos lados, rellena con null.

In [7]:
// FULL OUTER JOIN
val full = clientes.join(pedidos, clientes("id") === pedidos("clienteId"), "full")

println("=== FULL OUTER JOIN ===")
full.show()
println(s"Filas: ${full.count()}")

=== FULL OUTER JOIN ===
+---+------+----+---------+-------+
| id|nombre| ped|clienteId|importe|
+---+------+----+---------+-------+
|  1|   Ana|  P1|        1|  200.0|
|  1|   Ana|  P2|        1|   80.0|
|  2| Borja|  P3|        2|  350.0|
|  3|Carmen|NULL|     NULL|   NULL|
+---+------+----+---------+-------+

Filas: 4


full: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 3 more fields]

¿Cuánto pagaría cada cliente si comprara cada uno de los pedidos existentes? Genera todas las 

combinaciones posibles.

Técnica: CROSS JOIN — genera el producto cartesiano: cada fila del primer DataFrame combinada con 

cada fila del segundo. Requiere spark.sql.crossJoin.enabled = true en Spark 4.x.

In [8]:
// CROSS JOIN — en Spark 4.x se usa el tipo "cross" dentro de .join()
// Requiere spark.sql.crossJoin.enabled = true
// En Spark 4.x el cross join se expresa con lit(true) como condición y tipo "cross"
// La configuración spark.sql.crossJoin.enabled = true ya está activada en la Celda 1
val cross = clientes.join(pedidos, lit(true), "cross")

println("=== CROSS JOIN ===")
cross.show()
println(s"Filas: ${cross.count()} (${clientes.count()} clientes × ${pedidos.count()} pedidos)")

=== CROSS JOIN ===
+---+------+---+---------+-------+
| id|nombre|ped|clienteId|importe|
+---+------+---+---------+-------+
|  1|   Ana| P1|        1|  200.0|
|  2| Borja| P1|        1|  200.0|
|  3|Carmen| P1|        1|  200.0|
|  1|   Ana| P2|        1|   80.0|
|  2| Borja| P2|        1|   80.0|
|  3|Carmen| P2|        1|   80.0|
|  1|   Ana| P3|        2|  350.0|
|  2| Borja| P3|        2|  350.0|
|  3|Carmen| P3|        2|  350.0|
+---+------+---+---------+-------+

Filas: 9 (3 clientes × 3 pedidos)


cross: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 3 more fields]

// CROSS JOIN — en Spark 4.x se usa el tipo "cross" dentro de .join()
// Requiere spark.sql.crossJoin.enabled = true
// En Spark 4.x el cross join se expresa con lit(true) como condición y tipo "cross"
// La configuración spark.sql.crossJoin.enabled = true ya está activada en la Celda 1
val cross = clientes.join(pedidos, lit(true), "cross")

println("=== CROSS JOIN ===")
cross.show()
println(s"Filas: ${cross.count()} (${clientes.count()} clientes × ${pedidos.count()} pedidos)")

In [9]:
// LEFT SEMI JOIN
val semi = clientes.join(pedidos, clientes("id") === pedidos("clienteId"), "semi")

println("=== LEFT SEMI JOIN ===")
semi.show()
println(s"Filas: ${semi.count()}")

=== LEFT SEMI JOIN ===
+---+------+
| id|nombre|
+---+------+
|  1|   Ana|
|  2| Borja|
+---+------+

Filas: 2


semi: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string]

¿Qué clientes no han realizado ningún pedido todavía?
Técnica: LEFT ANTI JOIN — devuelve solo las filas del DataFrame izquierdo (clientes) que no tienen ninguna coincidencia en el derecho (pedidos). Es el complemento exacto del SEMI JOIN.

In [10]:
// LEFT ANTI JOIN
val anti = clientes.join(pedidos, clientes("id") === pedidos("clienteId"), "anti")

println("=== LEFT ANTI JOIN ===")
anti.show()
println(s"Filas: ${anti.count()}")

=== LEFT ANTI JOIN ===
+---+------+
| id|nombre|
+---+------+
|  3|Carmen|
+---+------+

Filas: 1


anti: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string]

Ejemplos

In [11]:
// DataFrame A — clientes
// Columnas: id, nombre
val clientes = Seq(
  (1, "Ana"),
  (2, "Borja"),
  (3, "Carmen")
).toDF("id", "nombre")

// DataFrame B — pedidos
// Columnas: id, clienteId, importe
// ATENCIÓN: también tiene una columna llamada "id" (el id del pedido)
val pedidos = Seq(
  (101, 1, 200.0),
  (102, 1,  80.0),
  (103, 2, 350.0)
).toDF("id", "clienteId", "importe")

println("=== DataFrame A: clientes ===")
clientes.show()
println("Columnas de clientes: " + clientes.columns.mkString(", "))

println("\n=== DataFrame B: pedidos ===")
pedidos.show()
println("Columnas de pedidos: " + pedidos.columns.mkString(", "))

println("\n⚠️  CONFLICTO: ambos DataFrames tienen una columna llamada 'id'")

=== DataFrame A: clientes ===
+---+------+
| id|nombre|
+---+------+
|  1|   Ana|
|  2| Borja|
|  3|Carmen|
+---+------+

Columnas de clientes: id, nombre

=== DataFrame B: pedidos ===
+---+---------+-------+
| id|clienteId|importe|
+---+---------+-------+
|101|        1|  200.0|
|102|        1|   80.0|
|103|        2|  350.0|
+---+---------+-------+

Columnas de pedidos: id, clienteId, importe

⚠️  CONFLICTO: ambos DataFrames tienen una columna llamada 'id'


clientes: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string]
pedidos: org.apache.spark.sql.package.DataFrame = [id: int, clienteId: int ... 1 more field]

In [12]:
//Hacemos el join sin resolver el conflicto

// Hacemos el join sin resolver el conflicto
val resultado = clientes.join(pedidos, clientes("id") === pedidos("clienteId"))

// Inspeccionamos el schema del resultado: veremos DOS columnas llamadas "id"
println("=== Schema tras el join (sin resolver el conflicto) ===")
resultado.printSchema()
println("Columnas del resultado: " + resultado.columns.mkString(", "))
println()

// Mostramos los datos — esto funciona porque show() no selecciona por nombre
println("=== Datos (show funciona porque no selecciona por nombre) ===")
resultado.show()

=== Schema tras el join (sin resolver el conflicto) ===
root
 |-- id: integer (nullable = false)
 |-- nombre: string (nullable = true)
 |-- id: integer (nullable = false)
 |-- clienteId: integer (nullable = false)
 |-- importe: double (nullable = false)

Columnas del resultado: id, nombre, id, clienteId, importe

=== Datos (show funciona porque no selecciona por nombre) ===
+---+------+---+---------+-------+
| id|nombre| id|clienteId|importe|
+---+------+---+---------+-------+
|  1|   Ana|102|        1|   80.0|
|  1|   Ana|101|        1|  200.0|
|  2| Borja|103|        2|  350.0|
+---+------+---+---------+-------+



resultado: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 3 more fields]

In [13]:
//Intentamos seleccionar "id" por nombre → ESTO PROVOCA EL ERROR
// Intentamos seleccionar "id" por nombre → ESTO PROVOCA EL ERROR
// Ejecuta esta celda para ver el mensaje de error de Spark
try {
  resultado.select("id").show()
} catch {
  case e: Exception =>
    println("❌ ERROR CAPTURADO:")
    println(e.getMessage.split("\n").head)
}

❌ ERROR CAPTURADO:
[AMBIGUOUS_REFERENCE] Reference `id` is ambiguous, could be: [`id`, `id`]. SQLSTATE: 42704


Broadcast Join: Optimización para Tablas Pequeñas

In [15]:
// ─────────────────────────────────────────────────
// TABLA GRANDE: pedidos
// En producción tendría millones de filas.
// Aquí usamos 12 filas para que el ejemplo sea legible.
// ─────────────────────────────────────────────────
val pedidos = Seq(
  ("P001", 1, "CAT-A", 120.0),
  ("P002", 1, "CAT-B",  45.0),
  ("P003", 2, "CAT-A", 200.0),
  ("P004", 2, "CAT-C",  89.0),
  ("P005", 3, "CAT-B", 310.0),
  ("P006", 3, "CAT-A",  55.0),
  ("P007", 4, "CAT-C", 175.0),
  ("P008", 4, "CAT-B",  30.0),
  ("P009", 5, "CAT-A", 250.0),
  ("P010", 5, "CAT-C",  99.0),
  ("P011", 6, "CAT-B", 145.0),
  ("P012", 6, "CAT-A",  70.0)
).toDF("pedidoId", "clienteId", "categoriaId", "importe")

// ─────────────────────────────────────────────────
// TABLA PEQUEÑA: categorías
// Solo 3 filas — catálogo estático que raramente cambia.
// Esta es la candidata perfecta para broadcast.
// ─────────────────────────────────────────────────
val categorias = Seq(
  ("CAT-A", "Electrónica",  "Alta gama"),
  ("CAT-B", "Hogar",        "Básica"),
  ("CAT-C", "Deportes",     "Media")
).toDF("id", "nombreCategoria", "segmento")

println("=== Tabla GRANDE: pedidos ===")
pedidos.show()
println(s"Filas en pedidos:    ${pedidos.count()}")

println("\n=== Tabla PEQUEÑA: categorias ===")
categorias.show()
println(s"Filas en categorias: ${categorias.count()}")

println("\n💡 'categorias' es la candidata al broadcast: pocas filas, datos estáticos.")

=== Tabla GRANDE: pedidos ===
+--------+---------+-----------+-------+
|pedidoId|clienteId|categoriaId|importe|
+--------+---------+-----------+-------+
|    P001|        1|      CAT-A|  120.0|
|    P002|        1|      CAT-B|   45.0|
|    P003|        2|      CAT-A|  200.0|
|    P004|        2|      CAT-C|   89.0|
|    P005|        3|      CAT-B|  310.0|
|    P006|        3|      CAT-A|   55.0|
|    P007|        4|      CAT-C|  175.0|
|    P008|        4|      CAT-B|   30.0|
|    P009|        5|      CAT-A|  250.0|
|    P010|        5|      CAT-C|   99.0|
|    P011|        6|      CAT-B|  145.0|
|    P012|        6|      CAT-A|   70.0|
+--------+---------+-----------+-------+

Filas en pedidos:    12

=== Tabla PEQUEÑA: categorias ===
+-----+---------------+---------+
|   id|nombreCategoria| segmento|
+-----+---------------+---------+
|CAT-A|    Electrónica|Alta gama|
|CAT-B|          Hogar|   Básica|
|CAT-C|       Deportes|    Media|
+-----+---------------+---------+

Filas en catego

pedidos: org.apache.spark.sql.package.DataFrame = [pedidoId: string, clienteId: int ... 2 more fields]
categorias: org.apache.spark.sql.package.DataFrame = [id: string, nombreCategoria: string ... 1 more field]

In [16]:
//Configuración del umbral automático
import org.apache.spark.sql.functions.broadcast
// ── Desactivamos el broadcast automático para ver el plan sin optimización ──
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

val umbralActual = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
println(s"Umbral autoBroadcastJoinThreshold: $umbralActual")
println("(valor -1 = broadcast automático DESACTIVADO)")
println()
println("Ahora Spark usará SortMergeJoin aunque la tabla sea pequeña.")
println("Esto nos permitirá ver la diferencia en el plan de ejecución.")

Umbral autoBroadcastJoinThreshold: -1
(valor -1 = broadcast automático DESACTIVADO)

Ahora Spark usará SortMergeJoin aunque la tabla sea pequeña.
Esto nos permitirá ver la diferencia en el plan de ejecución.


import org.apache.spark.sql.functions.broadcast
umbralActual: String = "-1"

In [17]:
Umbral autoBroadcastJoinThreshold: -1
(valor -1 = broadcast automático DESACTIVADO)

Ahora Spark usará SortMergeJoin aunque la tabla sea pequeña.
Esto nos permitirá ver la diferencia en el plan de ejecución.

(console):5:62 expected (`this` | Id)
Umbral autoBroadcastJoinThreshold: -1
(valor -1 = broadcast automático DESACTIVADO)

Ahora Spark usará SortMergeJoin aunque la tabla sea pequeña.
Esto nos permitirá ver la diferencia en el plan de ejecución.
                                                                                                                                                                                                               ^

 Join SIN broadcast — plan de ejecución
Join normal (sin broadcast) → SortMergeJoin
Con el broadcast automático desactivado, Spark utiliza SortMergeJoin: ordena ambos DataFrames por la clave de join y los fusiona. Esto implica mover datos entre nodos (shuffle), que es la operación más costosa en un cluster distribuido.

Observa en el plan de ejecución la presencia de SortMergeJoin y los pasos de Exchange (shuffle) que lo preceden.

In [17]:
// JOIN NORMAL — sin broadcast (broadcast automático ya está desactivado)
val sinBroadcast = pedidos.join(
  categorias,
  pedidos("categoriaId") === categorias("id")
)

println("=== Resultado del join sin broadcast ===")
sinBroadcast.select(
  col("pedidoId"),
  col("clienteId"),
  col("nombreCategoria"),
  col("segmento"),
  col("importe")
).orderBy("pedidoId").show()

println("\n=== Plan de ejecución SIN broadcast ===")
println("(busca 'SortMergeJoin' y 'Exchange' en el plan)")
println("─" * 60)
sinBroadcast.explain()

=== Resultado del join sin broadcast ===
+--------+---------+---------------+---------+-------+
|pedidoId|clienteId|nombreCategoria| segmento|importe|
+--------+---------+---------------+---------+-------+
|    P001|        1|    Electrónica|Alta gama|  120.0|
|    P002|        1|          Hogar|   Básica|   45.0|
|    P003|        2|    Electrónica|Alta gama|  200.0|
|    P004|        2|       Deportes|    Media|   89.0|
|    P005|        3|          Hogar|   Básica|  310.0|
|    P006|        3|    Electrónica|Alta gama|   55.0|
|    P007|        4|       Deportes|    Media|  175.0|
|    P008|        4|          Hogar|   Básica|   30.0|
|    P009|        5|    Electrónica|Alta gama|  250.0|
|    P010|        5|       Deportes|    Media|   99.0|
|    P011|        6|          Hogar|   Básica|  145.0|
|    P012|        6|    Electrónica|Alta gama|   70.0|
+--------+---------+---------------+---------+-------+


=== Plan de ejecución SIN broadcast ===
(busca 'SortMergeJoin' y 'Exchange' e

sinBroadcast: org.apache.spark.sql.package.DataFrame = [pedidoId: string, clienteId: int ... 5 more fields]

Join CON broadcast — plan de ejecución
Join con broadcast() forzado → BroadcastHashJoin
Al envolver categorias con broadcast(), le decimos a Spark que envíe una copia completa de esa tabla a cada nodo del cluster. Cada nodo puede hacer el join localmente con su partición de pedidos sin mover ningún dato por la red. Esto elimina completamente el shuffle.

In [18]:
import org.apache.spark.sql.functions.broadcast
// JOIN CON BROADCAST FORZADO
// broadcast() le indica a Spark que envíe 'categorias' completo a cada nodo
val conBroadcast = pedidos.join(
  broadcast(categorias),            // ← aquí está la clave
  pedidos("categoriaId") === categorias("id")
)

println("=== Resultado del join CON broadcast ===")
conBroadcast.select(
  col("pedidoId"),
  col("clienteId"),
  col("nombreCategoria"),
  col("segmento"),
  col("importe")
).orderBy("pedidoId").show()

println("\n=== Plan de ejecución CON broadcast ===")
println("(busca 'BroadcastHashJoin' y 'BroadcastExchange' en el plan)")
println("─" * 60)
conBroadcast.explain()

=== Resultado del join CON broadcast ===
+--------+---------+---------------+---------+-------+
|pedidoId|clienteId|nombreCategoria| segmento|importe|
+--------+---------+---------------+---------+-------+
|    P001|        1|    Electrónica|Alta gama|  120.0|
|    P002|        1|          Hogar|   Básica|   45.0|
|    P003|        2|    Electrónica|Alta gama|  200.0|
|    P004|        2|       Deportes|    Media|   89.0|
|    P005|        3|          Hogar|   Básica|  310.0|
|    P006|        3|    Electrónica|Alta gama|   55.0|
|    P007|        4|       Deportes|    Media|  175.0|
|    P008|        4|          Hogar|   Básica|   30.0|
|    P009|        5|    Electrónica|Alta gama|  250.0|
|    P010|        5|       Deportes|    Media|   99.0|
|    P011|        6|          Hogar|   Básica|  145.0|
|    P012|        6|    Electrónica|Alta gama|   70.0|
+--------+---------+---------------+---------+-------+


=== Plan de ejecución CON broadcast ===
(busca 'BroadcastHashJoin' y 'Broadca

import org.apache.spark.sql.functions.broadcast
conBroadcast: org.apache.spark.sql.package.DataFrame = [pedidoId: string, clienteId: int ... 5 more fields]

Umbral automático: dejar que Spark decida
Configurar el umbral automático de broadcast
Además de forzarlo manualmente con broadcast(), puedes configurar el umbral por debajo del cual Spark aplica el broadcast de forma automática. Si un DataFrame tiene un tamaño estimado menor que este umbral, Spark lo broadcastea sin que tengas que indicarlo.

El valor por defecto en Spark es 10 MB. Puedes aumentarlo para tablas de dimensión más grandes.

In [20]:
import org.apache.spark.sql.functions.broadcast
// ── Activamos el broadcast automático con un umbral de 10 MB ──
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)

val umbralActivo = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
println(s"Umbral autoBroadcastJoinThreshold: $umbralActivo bytes (${10} MB)")
println()

// Ahora hacemos el join SIN broadcast() explícito
// Spark debería aplicarlo automáticamente porque 'categorias' es pequeña
val autobroadcast = pedidos.join(
  categorias,                                       // sin broadcast() explícito
  pedidos("categoriaId") === categorias("id")
)

println("=== Plan con broadcast AUTOMÁTICO (umbral = 10 MB) ===")
println("(Spark decide solo porque 'categorias' cabe en el umbral)")
println("─" * 60)
autobroadcast.explain()

println("\n✅ Resultado idéntico (mismas filas, misma lógica):")
println(s"  Filas sin broadcast:  ${sinBroadcast.count()}")
println(s"  Filas con broadcast:  ${conBroadcast.count()}")
println(s"  Filas auto broadcast: ${autobroadcast.count()}")
println("  Los tres resultados son iguales — solo cambia la estrategia interna.")

Umbral autoBroadcastJoinThreshold: 10485760 bytes (10 MB)

=== Plan con broadcast AUTOMÁTICO (umbral = 10 MB) ===
(Spark decide solo porque 'categorias' cabe en el umbral)
────────────────────────────────────────────────────────────
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [categoriaId#362], [id#377], Inner, BuildRight, false
   :- LocalTableScan [pedidoId#360, clienteId#361, categoriaId#362, importe#363]
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, true]),false), [plan_id=1715]
      +- LocalTableScan [id#377, nombreCategoria#378, segmento#379]



✅ Resultado idéntico (mismas filas, misma lógica):
  Filas sin broadcast:  12
  Filas con broadcast:  12
  Filas auto broadcast: 12
  Los tres resultados son iguales — solo cambia la estrategia interna.


import org.apache.spark.sql.functions.broadcast
umbralActivo: String = "10485760"
autobroadcast: org.apache.spark.sql.package.DataFrame = [pedidoId: string, clienteId: int ... 5 more fields]

Práctica
Contexto del ejercicio
Trabajamos con datos de LogiData S.A., una empresa de logística con tres fuentes de datos:

clientes — maestro de clientes registrados
pedidos — registro de pedidos (algunos con un clienteId que no existe en el maestro)
repartidores — repartidores asignados a cada pedido

In [21]:
// === CLIENTES ===
val clientes = Seq(
  (1, "Ana García",    "Madrid"),
  (2, "Borja Ruiz",    "Barcelona"),
  (3, "Carmen López",  "Sevilla"),
  (4, "David Mora",    "Valencia"),
  (5, "Elena Pardo",   "Bilbao")
).toDF("clienteId", "nombre", "ciudad")

// === PEDIDOS ===
// clienteId 6 no existe en el maestro de clientes → dato huérfano
val pedidos = Seq(
  ("P001", 1, "2024-01-10", 250.0, "R01"),
  ("P002", 1, "2024-01-15", 180.0, "R02"),
  ("P003", 2, "2024-01-20", 420.0, "R01"),
  ("P004", 3, "2024-02-01",  95.0, "R03"),
  ("P005", 3, "2024-02-14",  60.0, "R02"),
  ("P006", 6, "2024-02-20", 310.0, "R03")  // ← cliente 6 no existe
).toDF("pedidoId", "clienteId", "fecha", "importe", "repartidorId")

// === REPARTIDORES ===
val repartidores = Seq(
  ("R01", "Miguel Sanz",   "Zona Norte"),
  ("R02", "Laura Vega",    "Zona Sur"),
  ("R03", "Pablo Fuentes", "Zona Este")
).toDF("repartidorId", "nombreRep", "zona")

println("DataFrames creados:")
println(s"  clientes:     ${clientes.count()} filas")
println(s"  pedidos:      ${pedidos.count()} filas")
println(s"  repartidores: ${repartidores.count()} filas")

DataFrames creados:
  clientes:     5 filas
  pedidos:      6 filas
  repartidores: 3 filas


clientes: org.apache.spark.sql.package.DataFrame = [clienteId: int, nombre: string ... 1 more field]
pedidos: org.apache.spark.sql.package.DataFrame = [pedidoId: string, clienteId: int ... 3 more fields]
repartidores: org.apache.spark.sql.package.DataFrame = [repartidorId: string, nombreRep: string ... 1 more field]

In [22]:
//Queremos cruzar pedidos con clientes para ver el nombre del cliente de cada pedido.
// Solo nos interesan los pedidos cuyo cliente existe en el maestro

// Buena práctica: renombrar la columna conflictiva ANTES del join
val pedidosRen = pedidos.withColumnRenamed("clienteId", "pedido_clienteId")

val resultado_inner = clientes.join(
  pedidosRen,
  col("clienteId") === col("pedido_clienteId"),
  "inner"
).select(
  col("pedidoId"),
  col("nombre").alias("cliente"),
  col("ciudad"),
  col("fecha"),
  col("importe")
).orderBy("pedidoId")

println("=== INNER JOIN: pedidos con cliente conocido ===")
resultado_inner.show()
println(s"Total filas: ${resultado_inner.count()}")
// P006 desaparece porque clienteId=6 no existe → esperamos 5 filas

=== INNER JOIN: pedidos con cliente conocido ===
+--------+------------+---------+----------+-------+
|pedidoId|     cliente|   ciudad|     fecha|importe|
+--------+------------+---------+----------+-------+
|    P001|  Ana García|   Madrid|2024-01-10|  250.0|
|    P002|  Ana García|   Madrid|2024-01-15|  180.0|
|    P003|  Borja Ruiz|Barcelona|2024-01-20|  420.0|
|    P004|Carmen López|  Sevilla|2024-02-01|   95.0|
|    P005|Carmen López|  Sevilla|2024-02-14|   60.0|
+--------+------------+---------+----------+-------+

Total filas: 5


pedidosRen: org.apache.spark.sql.package.DataFrame = [pedidoId: string, pedido_clienteId: int ... 3 more fields]
resultado_inner: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [pedidoId: string, cliente: string ... 3 more fields]

In [23]:
//Todos los clientes, aunque no hayan pedido nada:
val resultado_left = clientes.join(
  pedidosRen,
  col("clienteId") === col("pedido_clienteId"),
  "left"
).select(
  col("clienteId"),
  col("nombre"),
  col("pedidoId"),
  col("importe")
).orderBy("clienteId", "pedidoId")

println("=== LEFT JOIN: todos los clientes (con o sin pedidos) ===")
resultado_left.show()
// David Mora y Elena Pardo aparecen con null en pedidoId e importe

=== LEFT JOIN: todos los clientes (con o sin pedidos) ===
+---------+------------+--------+-------+
|clienteId|      nombre|pedidoId|importe|
+---------+------------+--------+-------+
|        1|  Ana García|    P001|  250.0|
|        1|  Ana García|    P002|  180.0|
|        2|  Borja Ruiz|    P003|  420.0|
|        3|Carmen López|    P004|   95.0|
|        3|Carmen López|    P005|   60.0|
|        4|  David Mora|    NULL|   NULL|
|        5| Elena Pardo|    NULL|   NULL|
+---------+------------+--------+-------+



resultado_left: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [clienteId: int, nombre: string ... 2 more fields]

In [24]:
//Todos los pedidos, aunque el cliente no exista en el maestro:
val resultado_right = clientes.join(
  pedidosRen,
  col("clienteId") === col("pedido_clienteId"),
  "right"
).select(
  col("pedido_clienteId").alias("clienteId"),
  col("nombre"),
  col("pedidoId"),
  col("importe")
).orderBy("pedidoId")

println("=== RIGHT JOIN: todos los pedidos (incluido el huérfano P006) ===")
resultado_right.show()
// P006 aparece con null en nombre

=== RIGHT JOIN: todos los pedidos (incluido el huérfano P006) ===
+---------+------------+--------+-------+
|clienteId|      nombre|pedidoId|importe|
+---------+------------+--------+-------+
|        1|  Ana García|    P001|  250.0|
|        1|  Ana García|    P002|  180.0|
|        2|  Borja Ruiz|    P003|  420.0|
|        3|Carmen López|    P004|   95.0|
|        3|Carmen López|    P005|   60.0|
|        6|        NULL|    P006|  310.0|
+---------+------------+--------+-------+



resultado_right: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [clienteId: int, nombre: string ... 2 more fields]

In [25]:
// SEMI y ANTI JOIN
//Estos dos tipos filtran filas sin añadir columnas del segundo DataFrame.
// LEFT SEMI: clientes que SÍ tienen al menos un pedido
val clientesConPedidos = clientes.join(
  pedidosRen,
  col("clienteId") === col("pedido_clienteId"),
  "semi"
)

println("=== LEFT SEMI: clientes que han hecho algún pedido ===")
clientesConPedidos.show()
// Solo columnas de clientes; David Mora y Elena Pardo no aparecen

// LEFT ANTI: clientes que NUNCA han pedido
val clientesSinPedidos = clientes.join(
  pedidosRen,
  col("clienteId") === col("pedido_clienteId"),
  "anti"
)

println("=== LEFT ANTI: clientes sin ningún pedido ===")
clientesSinPedidos.show()
// Solo David Mora y Elena Pardo

=== LEFT SEMI: clientes que han hecho algún pedido ===
+---------+------------+---------+
|clienteId|      nombre|   ciudad|
+---------+------------+---------+
|        1|  Ana García|   Madrid|
|        2|  Borja Ruiz|Barcelona|
|        3|Carmen López|  Sevilla|
+---------+------------+---------+

=== LEFT ANTI: clientes sin ningún pedido ===
+---------+-----------+--------+
|clienteId|     nombre|  ciudad|
+---------+-----------+--------+
|        4| David Mora|Valencia|
|        5|Elena Pardo|  Bilbao|
+---------+-----------+--------+



clientesConPedidos: org.apache.spark.sql.package.DataFrame = [clienteId: int, nombre: string ... 1 more field]
clientesSinPedidos: org.apache.spark.sql.package.DataFrame = [clienteId: int, nombre: string ... 1 more field]

In [26]:
//JOIN de tres tablas
// Renombramos todas las columnas que pueden colisionar
val pedidos3 = pedidos
  .withColumnRenamed("clienteId",    "pedido_clienteId")
  .withColumnRenamed("repartidorId", "pedido_repartidorId")

val resultado_triple = pedidos3
  .join(clientes,     col("pedido_clienteId")    === col("clienteId"),    "inner")
  .join(repartidores, col("pedido_repartidorId") === col("repartidorId"), "inner")
  .select(
    col("pedidoId"),
    col("nombre").alias("cliente"),
    col("ciudad"),
    col("fecha"),
    col("importe"),
    col("nombreRep").alias("repartidor"),
    col("zona")
  )
  .orderBy("pedidoId")

println("=== JOIN TRIPLE: pedidos + clientes + repartidores ===")
resultado_triple.show(truncate = false)

=== JOIN TRIPLE: pedidos + clientes + repartidores ===
+--------+------------+---------+----------+-------+-------------+----------+
|pedidoId|cliente     |ciudad   |fecha     |importe|repartidor   |zona      |
+--------+------------+---------+----------+-------+-------------+----------+
|P001    |Ana García  |Madrid   |2024-01-10|250.0  |Miguel Sanz  |Zona Norte|
|P002    |Ana García  |Madrid   |2024-01-15|180.0  |Laura Vega   |Zona Sur  |
|P003    |Borja Ruiz  |Barcelona|2024-01-20|420.0  |Miguel Sanz  |Zona Norte|
|P004    |Carmen López|Sevilla  |2024-02-01|95.0   |Pablo Fuentes|Zona Este |
|P005    |Carmen López|Sevilla  |2024-02-14|60.0   |Laura Vega   |Zona Sur  |
+--------+------------+---------+----------+-------+-------------+----------+



pedidos3: org.apache.spark.sql.package.DataFrame = [pedidoId: string, pedido_clienteId: int ... 3 more fields]
resultado_triple: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [pedidoId: string, cliente: string ... 5 more fields]

In [27]:
//Comparamos el plan generado con y sin broadcast para ver cómo Spark lo optimiza:

val pedidos4 = pedidos
  .withColumnRenamed("repartidorId", "pedido_repartidorId")

// Sin broadcast
val sinBroadcast = pedidos4.join(
  repartidores,
  col("pedido_repartidorId") === col("repartidorId"),
  "inner"
)

println("=== Plan SIN broadcast ===")
sinBroadcast.explain()

// Con broadcast forzado (repartidores es pequeño)
val conBroadcast = pedidos4.join(
  broadcast(repartidores),
  col("pedido_repartidorId") === col("repartidorId"),
  "inner"
)

println("\n=== Plan CON broadcast ===")
conBroadcast.explain()

// Los resultados deben ser idénticos
println(s"\nFilas sin broadcast: ${sinBroadcast.count()}")
println(s"Filas con broadcast: ${conBroadcast.count()}")

=== Plan SIN broadcast ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [pedido_repartidorId#697], [repartidorId#571], Inner, BuildRight, false
   :- LocalTableScan [pedidoId#553, clienteId#554, fecha#555, importe#556, pedido_repartidorId#697]
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, true]),false), [plan_id=2507]
      +- LocalTableScan [repartidorId#571, nombreRep#572, zona#573]



=== Plan CON broadcast ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [pedido_repartidorId#697], [repartidorId#571], Inner, BuildRight, false
   :- LocalTableScan [pedidoId#553, clienteId#554, fecha#555, importe#556, pedido_repartidorId#697]
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, true]),false), [plan_id=2521]
      +- LocalTableScan [repartidorId#571, nombreRep#572, zona#573]



Filas sin broadcast: 6
Filas con broadcast: 6


pedidos4: org.apache.spark.sql.package.DataFrame = [pedidoId: string, clienteId: int ... 3 more fields]
sinBroadcast: org.apache.spark.sql.package.DataFrame = [pedidoId: string, clienteId: int ... 6 more fields]
conBroadcast: org.apache.spark.sql.package.DataFrame = [pedidoId: string, clienteId: int ... 6 more fields]

In [28]:
//Tenemos dos listas de ciudades con cobertura de LogiData en distintos meses

val ciudadesEnero = Seq(
  ("Madrid",    "Norte"),
  ("Barcelona", "Este"),
  ("Sevilla",   "Sur")
).toDF("ciudad", "zona")

val ciudadesFebrero = Seq(
  ("Madrid",   "Norte"),
  ("Valencia", "Este"),
  ("Bilbao",   "Norte")
).toDF("ciudad", "zona")

// UNION con duplicados (Madrid aparece dos veces)
val todasConDup = ciudadesEnero.union(ciudadesFebrero)
println(s"=== union (con duplicados): ${todasConDup.count()} filas ===")
todasConDup.show()

// UNION sin duplicados
val todasSinDup = ciudadesEnero.union(ciudadesFebrero).distinct()
println(s"=== union.distinct (sin duplicados): ${todasSinDup.count()} filas ===")
todasSinDup.show()

// EXCEPT: ciudades de enero que NO están en febrero
val soloEnero = ciudadesEnero.except(ciudadesFebrero)
println("=== except: ciudades solo cubiertas en enero ===")
soloEnero.show()
// Esperamos: Barcelona y Sevilla

=== union (con duplicados): 6 filas ===
+---------+-----+
|   ciudad| zona|
+---------+-----+
|   Madrid|Norte|
|Barcelona| Este|
|  Sevilla|  Sur|
|   Madrid|Norte|
| Valencia| Este|
|   Bilbao|Norte|
+---------+-----+

=== union.distinct (sin duplicados): 5 filas ===
+---------+-----+
|   ciudad| zona|
+---------+-----+
|   Madrid|Norte|
|Barcelona| Este|
|  Sevilla|  Sur|
| Valencia| Este|
|   Bilbao|Norte|
+---------+-----+

=== except: ciudades solo cubiertas en enero ===
+---------+----+
|   ciudad|zona|
+---------+----+
|Barcelona|Este|
|  Sevilla| Sur|
+---------+----+



ciudadesEnero: org.apache.spark.sql.package.DataFrame = [ciudad: string, zona: string]
ciudadesFebrero: org.apache.spark.sql.package.DataFrame = [ciudad: string, zona: string]
todasConDup: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, zona: string]
todasSinDup: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, zona: string]
soloEnero: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, zona: string]

In [29]:
println("=" * 52)
println("RESUMEN — Día 15 Sesión 2 | LogiData S.A.")
println("=" * 52)

val checks = Seq(
  ("INNER JOIN — pedidos con cliente válido",     resultado_inner.count()    == 5),
  ("LEFT ANTI  — clientes sin pedidos",           clientesSinPedidos.count() == 2),
  ("JOIN TRIPLE — pedidos+clientes+repartidores", resultado_triple.count()   == 5),
  ("UNION.DISTINCT — sin duplicados",             todasSinDup.count()        == 5),
  ("EXCEPT — solo ciudades de enero",             soloEnero.count()          == 2)
)

checks.foreach { case (desc, ok) =>
  println(s"${if (ok) "✅ CORRECTO" else "❌ REVISAR"} — $desc")
}

RESUMEN — Día 15 Sesión 2 | LogiData S.A.
✅ CORRECTO — INNER JOIN — pedidos con cliente válido
✅ CORRECTO — LEFT ANTI  — clientes sin pedidos
✅ CORRECTO — JOIN TRIPLE — pedidos+clientes+repartidores
✅ CORRECTO — UNION.DISTINCT — sin duplicados
✅ CORRECTO — EXCEPT — solo ciudades de enero


checks: Seq[(String, Boolean)] = List(
  ("INNER JOIN — pedidos con cliente válido", true),
  ("LEFT ANTI  — clientes sin pedidos", true),
  ("JOIN TRIPLE — pedidos+clientes+repartidores", true),
  ("UNION.DISTINCT — sin duplicados", true),
  ("EXCEPT — solo ciudades de enero", true)
)

Caso de Estudio - MediRed S.A.. Análisis de la Red Nacional de Farmacias
🏢 La empresa
MediRed S.A. es una red nacional de farmacias con presencia en cuatro comunidades autónomas. La empresa gestiona sus operaciones con varios sistemas de información que han crecido de forma independiente a lo largo de los años. Como resultado, los datos están fragmentados en distintas fuentes que nunca se han cruzado de forma sistemática.

El departamento de tecnología ha decidido construir un pipeline de análisis con Apache Spark para responder preguntas de negocio que hasta ahora eran imposibles de responder sin semanas de trabajo manual. Tú eres el ingeniero de datos encargado del proyecto.

In [30]:
// === FARMACIAS ===
val farmacias = Seq(
  (1,  "Farmacia Central",     "Madrid",    "Madrid",    "Laura Vega"),
  (2,  "Farmacia del Sol",     "Madrid",    "Madrid",    "Carlos Ruiz"),
  (3,  "Farmacia Diagonal",    "Barcelona", "Cataluña",  "Ana Puig"),
  (4,  "Farmacia Rambla",      "Barcelona", "Cataluña",  "Marta Font"),
  (5,  "Farmacia Norte",       "Bilbao",    "País Vasco", "Jon Etxea"),
  (6,  "Farmacia Ría",         "Bilbao",    "País Vasco", "Amaia Goñi"),
  (7,  "Farmacia Gran Vía",    "Sevilla",   "Andalucía", "Pedro Mora"),
  (8,  "Farmacia Triana",      "Sevilla",   "Andalucía", "Rosa Leal")
).toDF("farmaciaId", "nombre", "ciudad", "comunidad", "titular")

// === VENTAS ===
// farmaciaId 99 no existe en el maestro → venta huérfana del proveedor externo
val ventas = Seq(
  ("V001",  1, "P-AMOX", 3, 18.50, "2024-01-05"),
  ("V002",  1, "P-IBUP",10, 42.00, "2024-01-07"),
  ("V003",  2, "P-VITA", 5, 25.00, "2024-01-08"),
  ("V004",  3, "P-AMOX", 2, 12.50, "2024-01-10"),
  ("V005",  3, "P-PARA", 8, 16.00, "2024-01-12"),
  ("V006",  4, "P-IBUP", 6, 24.00, "2024-01-15"),
  ("V007",  5, "P-VITA",12, 60.00, "2024-01-18"),
  ("V008",  6, "P-PARA", 4,  8.00, "2024-01-20"),
  ("V009",  7, "P-AMOX", 7, 43.75, "2024-02-01"),
  ("V010",  7, "P-IBUP", 3, 12.00, "2024-02-03"),
  ("V011",  8, "P-VITA", 9, 45.00, "2024-02-05"),
  ("V012", 99, "P-PARA", 2,  4.00, "2024-02-10")  // ← farmaciaId 99 no existe
).toDF("ventaId", "farmaciaId", "productoId", "cantidad", "importe", "fecha")

// === PRODUCTOS (tabla pequeña: 4 registros) ===
val productos = Seq(
  ("P-AMOX", "Amoxicilina 500mg",  "Antibiótico",  true),
  ("P-IBUP", "Ibuprofeno 600mg",   "Analgésico",   false),
  ("P-VITA", "Vitamina D 1000 UI", "Vitamina",     false),
  ("P-PARA", "Paracetamol 1g",     "Analgésico",   false)
).toDF("productoId", "nombreProducto", "categoria", "requiereReceta")

// === INSPECCIONES Q1 (enero-marzo) ===
val inspeccionesQ1 = Seq(
  (1, "2024-01-15", "Apto"),
  (2, "2024-01-22", "Apto"),
  (3, "2024-02-05", "No Apto"),
  (5, "2024-02-18", "Apto"),
  (7, "2024-03-10", "Apto")
).toDF("farmaciaId", "fecha", "resultado")

// === INSPECCIONES Q2 (abril-junio) ===
val inspeccionesQ2 = Seq(
  (3, "2024-04-08", "Apto"),    // corrigió el No Apto del Q1
  (4, "2024-04-20", "Apto"),
  (6, "2024-05-12", "No Apto"),
  (7, "2024-05-25", "Apto"),
  (8, "2024-06-03", "Apto")
).toDF("farmaciaId", "fecha", "resultado")

println("Datos cargados:")
println(s"  farmacias:      ${farmacias.count()} registros")
println(s"  ventas:         ${ventas.count()} registros")
println(s"  productos:      ${productos.count()} registros")
println(s"  inspeccionesQ1: ${inspeccionesQ1.count()} registros")
println(s"  inspeccionesQ2: ${inspeccionesQ2.count()} registros")

Datos cargados:
  farmacias:      8 registros
  ventas:         12 registros
  productos:      4 registros
  inspeccionesQ1: 5 registros
  inspeccionesQ2: 5 registros


farmacias: org.apache.spark.sql.package.DataFrame = [farmaciaId: int, nombre: string ... 3 more fields]
ventas: org.apache.spark.sql.package.DataFrame = [ventaId: string, farmaciaId: int ... 4 more fields]
productos: org.apache.spark.sql.package.DataFrame = [productoId: string, nombreProducto: string ... 2 more fields]
inspeccionesQ1: org.apache.spark.sql.package.DataFrame = [farmaciaId: int, fecha: string ... 1 more field]
inspeccionesQ2: org.apache.spark.sql.package.DataFrame = [farmaciaId: int, fecha: string ... 1 more field]

In [ ]:
//Cada misión plantea una pregunta de negocio real.
 Para cada una, elige el tipo de join o la operación de conjunto adecuada,
  implementa la solución y obtén el resultado esperado.


Misión 1 — Informe de ventas con nombre de producto
Petición del negocio: "Necesitamos un informe de ventas que muestre el nombre del producto y su categoría, no solo el código. Solo queremos las ventas de productos que existan en nuestro catálogo."

Lo que debes hacer:

Cruza ventas con productos para obtener un informe enriquecido. Muestra las columnas: ventaId, nombreProducto, categoria, requiereReceta, cantidad, importe, ordenado por importe descendente.

Pistas:

Ambos DataFrames tienen una columna productoId. Renómbrala antes del join para evitar ambigüedad.
El enunciado dice "solo ventas de productos que existan en el catálogo" → ¿qué tipo de join es ese?
productos tiene solo 4 filas. ¿Tiene sentido aplicar alguna optimización de rendimiento?
Resultado aproximado: 12 filas (todas las ventas tienen producto válido en este dataset)

In [34]:
val ventas2 = ventas
  .withColumnRenamed("productoId",    "id_producto_venta")

val informe_ventas = ventas2
  .join(productos,     col("id_producto_venta")    === col("productoId"),    "inner")
  .select(
    col("ventaId"),
    col("nombreProducto"),
    col("categoria"),
    col("requiereReceta"),
    col("cantidad"),
    col("importe"),
  )
  .orderBy(col("importe").desc)

println("=== Informe de Ventas ===")
informe_ventas.show(truncate = false)

=== Informe de Ventas ===
+-------+------------------+-----------+--------------+--------+-------+
|ventaId|nombreProducto    |categoria  |requiereReceta|cantidad|importe|
+-------+------------------+-----------+--------------+--------+-------+
|V007   |Vitamina D 1000 UI|Vitamina   |false         |12      |60.0   |
|V011   |Vitamina D 1000 UI|Vitamina   |false         |9       |45.0   |
|V009   |Amoxicilina 500mg |Antibiótico|true          |7       |43.75  |
|V002   |Ibuprofeno 600mg  |Analgésico |false         |10      |42.0   |
|V003   |Vitamina D 1000 UI|Vitamina   |false         |5       |25.0   |
|V006   |Ibuprofeno 600mg  |Analgésico |false         |6       |24.0   |
|V001   |Amoxicilina 500mg |Antibiótico|true          |3       |18.5   |
|V005   |Paracetamol 1g    |Analgésico |false         |8       |16.0   |
|V004   |Amoxicilina 500mg |Antibiótico|true          |2       |12.5   |
|V010   |Ibuprofeno 600mg  |Analgésico |false         |3       |12.0   |
|V008   |Paracetamol 1g  

ventas2: org.apache.spark.sql.package.DataFrame = [ventaId: string, farmaciaId: int ... 4 more fields]
informe_ventas: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ventaId: string, nombreProducto: string ... 4 more fields]

In [35]:
//otra forma optimizada
import org.apache.spark.sql.functions.{col, broadcast}

// 1. Renombrar para evitar ambigüedad (como ya estabas haciendo)
val ventas2 = ventas.withColumnRenamed("productoId", "id_producto_venta")

// 2. Join con optimización broadcast y selección correcta
val informe_ventas = ventas2
  .join(
    broadcast(productos), 
    col("id_producto_venta") === col("productoId"), 
    "inner"
  )
  .select(
    col("ventaId"),
    col("nombreProducto"),
    col("categoria"),
    col("requiereReceta"),
    col("cantidad"),
    col("importe")
  )
  // 3. Ordenar por importe descendente
  .orderBy(col("importe").desc)

// Mostrar resultado
println("=== Informe de Ventas ===")
informe_ventas.show(truncate = false)


=== Informe de Ventas ===
+-------+------------------+-----------+--------------+--------+-------+
|ventaId|nombreProducto    |categoria  |requiereReceta|cantidad|importe|
+-------+------------------+-----------+--------------+--------+-------+
|V007   |Vitamina D 1000 UI|Vitamina   |false         |12      |60.0   |
|V011   |Vitamina D 1000 UI|Vitamina   |false         |9       |45.0   |
|V009   |Amoxicilina 500mg |Antibiótico|true          |7       |43.75  |
|V002   |Ibuprofeno 600mg  |Analgésico |false         |10      |42.0   |
|V003   |Vitamina D 1000 UI|Vitamina   |false         |5       |25.0   |
|V006   |Ibuprofeno 600mg  |Analgésico |false         |6       |24.0   |
|V001   |Amoxicilina 500mg |Antibiótico|true          |3       |18.5   |
|V005   |Paracetamol 1g    |Analgésico |false         |8       |16.0   |
|V004   |Amoxicilina 500mg |Antibiótico|true          |2       |12.5   |
|V010   |Ibuprofeno 600mg  |Analgésico |false         |3       |12.0   |
|V008   |Paracetamol 1g  

import org.apache.spark.sql.functions.{col, broadcast}
ventas2: org.apache.spark.sql.package.DataFrame = [ventaId: string, farmaciaId: int ... 4 more fields]
informe_ventas: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ventaId: string, nombreProducto: string ... 4 more fields]

Misión 2 — Farmacias sin ventas registradas
Petición del negocio: "Queremos saber qué farmacias de la red no tienen ninguna venta registrada en el sistema. Puede indicar un problema de integración con su sistema de caja."

Lo que debes hacer:

Obtén la lista de farmacias del maestro que no tienen ninguna venta en el DataFrame ventas. Muestra farmaciaId, nombre y ciudad.

Pistas:

Renombra la columna farmaciaId de ventas antes del join.
El enunciado dice "que NO tienen ninguna venta" → ¿qué tipo de join devuelve solo las filas del lado izquierdo que no tienen coincidencia?
Resultado esperado: En el dataset de ejemplo, ninguna farmacia debería quedar sin ventas. Si el resultado es 0 filas, está correcto.

In [36]:
// 1. Renombrar farmaciaId en ventas para evitar colisiones
val ventasParaJoin = ventas.withColumnRenamed("farmaciaId", "id_farmacia_venta")

// 2. Realizar un Left Anti Join
// Esto devolverá solo las farmacias que NO están en ventas
val farmaciasSinVentas = farmacias.join(
  ventasParaJoin,
  col("farmaciaId") === col("id_farmacia_venta"),
  "left_anti"
)
.select(
  col("farmaciaId"),
  col("nombre"),
  col("ciudad")
)

// 3. Mostrar resultado
println("=== Farmacias sin ventas registradas ===")
farmaciasSinVentas.show(truncate = false)


=== Farmacias sin ventas registradas ===
+----------+------+------+
|farmaciaId|nombre|ciudad|
+----------+------+------+
+----------+------+------+



ventasParaJoin: org.apache.spark.sql.package.DataFrame = [ventaId: string, id_farmacia_venta: int ... 4 more fields]
farmaciasSinVentas: org.apache.spark.sql.package.DataFrame = [farmaciaId: int, nombre: string ... 1 more field]

Misión 3 — Detectar ventas huérfanas
Petición del negocio: "El equipo de auditoría necesita localizar todas las ventas cuyo farmaciaId no existe en nuestro maestro de farmacias. Estos registros vienen del proveedor externo y hay que investigarlos."

Lo que debes hacer:

Encuentra las ventas cuyo farmaciaId no tiene correspondencia en el maestro farmacias. Muestra ventaId, farmaciaId, productoId e importe.

Pistas:

Esta vez la relación se invierte: quieres filas de ventas que no tienen pareja en farmacias.
Recuerda que los joins semi y anti operan siempre sobre el lado izquierdo. ¿Cuál de los dos DataFrames debe ir a la izquierda?
Resultado esperado: 1 fila — la venta V012 con farmaciaId = 99.

In [37]:
import org.apache.spark.sql.functions.col

// 1. Preparamos el maestro de farmacias (opcionalmente renombramos para claridad)
val farmaciasMaestro = farmacias.select(col("farmaciaId").as("id_maestro"))

// 2. Ejecutamos el Left Anti Join
// Ponemos 'ventas' a la izquierda porque es de donde queremos extraer los huérfanos
val ventasHuerfanas = ventas.join(
    farmaciasMaestro, 
    col("farmaciaId") === col("id_maestro"), 
    "left_anti"
  )
  .select(
    col("ventaId"), 
    col("farmaciaId"), 
    col("productoId"), 
    col("importe")
  )

// 3. Mostrar resultado
println("=== Ventas Huérfanas Detectadas ===")
ventasHuerfanas.show(truncate = false)


=== Ventas Huérfanas Detectadas ===
+-------+----------+----------+-------+
|ventaId|farmaciaId|productoId|importe|
+-------+----------+----------+-------+
|V012   |99        |P-PARA    |4.0    |
+-------+----------+----------+-------+



import org.apache.spark.sql.functions.col
farmaciasMaestro: org.apache.spark.sql.package.DataFrame = [id_maestro: int]
ventasHuerfanas: org.apache.spark.sql.package.DataFrame = [ventaId: string, farmaciaId: int ... 2 more fields]

Misión 4 — Informe completo: ventas + farmacia + producto
Petición del negocio: "Necesitamos un informe ejecutivo que cruce las tres fuentes: cada venta con el nombre de la farmacia, su comunidad autónoma y el nombre del producto. Solo queremos ventas con farmacia y producto válidos."

Lo que debes hacer:

Encadena dos joins para obtener el informe completo. Muestra: ventaId, nombre (farmacia), comunidad, nombreProducto, categoria, cantidad, importe, fecha. Ordena por comunidad y luego por importe descendente.

Pistas:

Tendrás que renombrar varias columnas antes de los joins (hay columnas farmaciaId y productoId en más de un DataFrame).
Encadena los dos joins uno tras otro sobre el mismo pipeline.
"Solo ventas con farmacia y producto válidos" → ¿qué tipo de join para cada cruce?
Resultado esperado: 11 filas (V012 queda excluida por tener farmaciaId inválido).

In [38]:
import org.apache.spark.sql.functions.col

// 1. Preparar DataFrames renombrando columnas clave para evitar ambigüedad
val v = ventas.withColumnRenamed("farmaciaId", "v_fId").withColumnRenamed("productoId", "v_pId")
val f = farmacias.withColumnRenamed("farmaciaId", "f_fId")
val p = productos.withColumnRenamed("productoId", "p_pId")

// 2. Encadenar los Joins (Inner por defecto para ventas válidas)
val informeEjecutivo = v
  // Primer cruce: Ventas + Farmacias
  .join(f, col("v_fId") === col("f_fId"), "inner")
  // Segundo cruce: Resultado anterior + Productos
  .join(p, col("v_pId") === col("p_pId"), "inner")
  // 3. Selección de columnas solicitadas
  .select(
    col("ventaId"),
    col("nombre").as("nombreFarmacia"),
    col("comunidad"),
    col("nombreProducto"),
    col("categoria"),
    col("cantidad"),
    col("importe"),
    col("fecha")
  )
  // 4. Ordenar por comunidad (asc) e importe (desc)
  .orderBy(col("comunidad").asc, col("importe").desc)

// Mostrar el resultado
println("=== Informe Ejecutivo de Ventas (Válidas) ===")
informeEjecutivo.show(truncate = false)


=== Informe Ejecutivo de Ventas (Válidas) ===
+-------+-----------------+----------+------------------+-----------+--------+-------+----------+
|ventaId|nombreFarmacia   |comunidad |nombreProducto    |categoria  |cantidad|importe|fecha     |
+-------+-----------------+----------+------------------+-----------+--------+-------+----------+
|V011   |Farmacia Triana  |Andalucía |Vitamina D 1000 UI|Vitamina   |9       |45.0   |2024-02-05|
|V009   |Farmacia Gran Vía|Andalucía |Amoxicilina 500mg |Antibiótico|7       |43.75  |2024-02-01|
|V010   |Farmacia Gran Vía|Andalucía |Ibuprofeno 600mg  |Analgésico |3       |12.0   |2024-02-03|
|V006   |Farmacia Rambla  |Cataluña  |Ibuprofeno 600mg  |Analgésico |6       |24.0   |2024-01-15|
|V005   |Farmacia Diagonal|Cataluña  |Paracetamol 1g    |Analgésico |8       |16.0   |2024-01-12|
|V004   |Farmacia Diagonal|Cataluña  |Amoxicilina 500mg |Antibiótico|2       |12.5   |2024-01-10|
|V002   |Farmacia Central |Madrid    |Ibuprofeno 600mg  |Analgésico |10 

import org.apache.spark.sql.functions.col
v: org.apache.spark.sql.package.DataFrame = [ventaId: string, v_fId: int ... 4 more fields]
f: org.apache.spark.sql.package.DataFrame = [f_fId: int, nombre: string ... 3 more fields]
p: org.apache.spark.sql.package.DataFrame = [p_pId: string, nombreProducto: string ... 2 more fields]
informeEjecutivo: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ventaId: string, nombreFarmacia: string ... 6 more fields]

Misión 5 — Farmacias que han sido inspeccionadas en ambos trimestres
Petición del negocio: "Sanidad nos pide la lista de farmacias que han pasado inspección tanto en Q1 como en Q2. Son las que tienen seguimiento completo este año."

Lo que debes hacer:

Obtén los farmaciaId que aparecen tanto en inspeccionesQ1 como en inspeccionesQ2. Luego cruza ese resultado con el maestro farmacias para mostrar el nombre y la ciudad.

Pistas:

Primero extrae solo la columna farmaciaId de cada DataFrame de inspecciones antes de operar.
¿Qué operación de conjunto devuelve los elementos que existen en los dos conjuntos?
Después, para añadir el nombre de la farmacia, necesitarás un join adicional.
Resultado esperado: Las farmacias 3 y 7 (Farmacia Diagonal y Farmacia Gran Vía).

In [39]:
import org.apache.spark.sql.functions.col

// 1. Extraer solo los IDs de cada trimestre
val idsQ1 = inspeccionesQ1.select("farmaciaId")
val idsQ2 = inspeccionesQ2.select("farmaciaId")

// 2. Operación de conjunto: Intersección (solo los que están en AMBOS)
val farmaciasAmbosTrimestres = idsQ1.intersect(idsQ2)

// 3. Join con el maestro para obtener nombre y ciudad
val informeInspeccionesCompletas = farmaciasAmbosTrimestres
  .join(farmacias, "farmaciaId") // Join simplificado usando el nombre de columna común
  .select(
    col("farmaciaId"),
    col("nombre"),
    col("ciudad")
  )

// 4. Mostrar resultado
println("=== Farmacias con seguimiento completo (Q1 + Q2) ===")
informeInspeccionesCompletas.show(truncate = false)


=== Farmacias con seguimiento completo (Q1 + Q2) ===
+----------+-----------------+---------+
|farmaciaId|nombre           |ciudad   |
+----------+-----------------+---------+
|3         |Farmacia Diagonal|Barcelona|
|7         |Farmacia Gran Vía|Sevilla  |
+----------+-----------------+---------+



import org.apache.spark.sql.functions.col
idsQ1: org.apache.spark.sql.package.DataFrame = [farmaciaId: int]
idsQ2: org.apache.spark.sql.package.DataFrame = [farmaciaId: int]
farmaciasAmbosTrimestres: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [farmaciaId: int]
informeInspeccionesCompletas: org.apache.spark.sql.package.DataFrame = [farmaciaId: int, nombre: string ... 1 more field]

Misión 6 — Historial completo de inspecciones del año
Petición del negocio: "Queremos un registro unificado de todas las inspecciones realizadas durante el año, tanto del Q1 como del Q2, ordenado por fecha."

Lo que debes hacer:

Combina inspeccionesQ1 e inspeccionesQ2 en un único DataFrame con todas las inspecciones. Asegúrate de que no haya duplicados. Muestra farmaciaId, fecha, resultado, ordenado por fecha.

Pistas:

Ambos DataFrames tienen exactamente las mismas columnas en el mismo orden.
union solo apila filas sin eliminar repetidos. ¿Necesitas hacer algo más?
Comprueba el número de filas antes y después de eliminar duplicados para ver si había alguno.
Resultado aproximado: 10 filas únicas (5 de Q1 + 5 de Q2, sin solapamiento en este dataset).



In [41]:


// 1. Unificar los dos DataFrames (Q1 y Q2)
// 'union' apila los registros. 'distinct' elimina filas idénticas si las hubiera.
val historialInspecciones = inspeccionesQ1
  .union(inspeccionesQ2)
  .distinct()

// 2. Aplicar ordenamiento por fecha
val historialOrdenado = historialInspecciones
  .orderBy(col("fecha").asc)

// 3. Mostrar resultado
println("=== Historial Unificado de Inspecciones 2024 ===")
historialOrdenado.show(truncate = false)

// Opcional: Verificar conteo
println(s"Total inspecciones registradas: ${historialOrdenado.count()}")


=== Historial Unificado de Inspecciones 2024 ===
+----------+----------+---------+
|farmaciaId|fecha     |resultado|
+----------+----------+---------+
|1         |2024-01-15|Apto     |
|2         |2024-01-22|Apto     |
|3         |2024-02-05|No Apto  |
|5         |2024-02-18|Apto     |
|7         |2024-03-10|Apto     |
|3         |2024-04-08|Apto     |
|4         |2024-04-20|Apto     |
|6         |2024-05-12|No Apto  |
|7         |2024-05-25|Apto     |
|8         |2024-06-03|Apto     |
+----------+----------+---------+

Total inspecciones registradas: 10


historialInspecciones: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [farmaciaId: int, fecha: string ... 1 more field]
historialOrdenado: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [farmaciaId: int, fecha: string ... 1 more field]

Misión 7 — Farmacias inspeccionadas solo en Q1 (sin revisión en Q2)
Petición del negocio: "Necesitamos contactar con las farmacias que fueron inspeccionadas en Q1 pero que todavía no tienen inspección registrada en Q2. Hay que planificar su visita."

Lo que debes hacer:

Encuentra los farmaciaId que aparecen en inspeccionesQ1 pero no en inspeccionesQ2. Muestra nombre y titular de cada farmacia.

Pistas:

Extrae solo la columna farmaciaId de cada DataFrame de inspecciones antes de operar.
¿Qué operación de conjunto devuelve los elementos del primero que no están en el segundo?
Después cruza con farmacias para añadir nombre y titular.
Resultado esperado: Farmacias 1, 2 y 5 (Central, del Sol y Norte).

In [42]:
// 1. Extraer solo los IDs de farmacia de cada trimestre
val idsQ1 = inspeccionesQ1.select("farmaciaId")
val idsQ2 = inspeccionesQ2.select("farmaciaId")

// 2. Operación de conjunto: Diferencia
// Obtenemos IDs que están en Q1 pero NO en Q2
val pendientesRevisionQ2 = idsQ1.except(idsQ2)

// 3. Join con el maestro para obtener datos de contacto (Nombre y Titular)
val informeContactos = pendientesRevisionQ2
  .join(farmacias, "farmaciaId")
  .select(
    col("farmaciaId"),
    col("nombre"),
    col("titular")
  )

// 4. Mostrar resultado
println("=== Farmacias pendientes de inspección en Q2 ===")
informeContactos.show(truncate = false)


=== Farmacias pendientes de inspección en Q2 ===
+----------+----------------+-----------+
|farmaciaId|nombre          |titular    |
+----------+----------------+-----------+
|1         |Farmacia Central|Laura Vega |
|2         |Farmacia del Sol|Carlos Ruiz|
|5         |Farmacia Norte  |Jon Etxea  |
+----------+----------------+-----------+



idsQ1: org.apache.spark.sql.package.DataFrame = [farmaciaId: int]
idsQ2: org.apache.spark.sql.package.DataFrame = [farmaciaId: int]
pendientesRevisionQ2: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [farmaciaId: int]
informeContactos: org.apache.spark.sql.package.DataFrame = [farmaciaId: int, nombre: string ... 1 more field]

Misión 8 — Plan de auditoría cruzado: todas las farmacias × todas las categorías de producto
Petición del negocio: "El equipo de cumplimiento quiere generar una matriz completa de todas las combinaciones posibles de farmacia y categoría de producto. Se usará como plantilla para el plan de auditoría anual."

Lo que debes hacer:

Genera el producto cartesiano entre farmacias (solo farmaciaId y nombre) y las categorías únicas de productos. Muestra farmaciaId, nombre y categoria, ordenado por farmaciaId.

Pistas:

Primero extrae las categorías únicas de productos con .select("categoria").distinct().
Recuerda que el cross join requiere spark.sql.crossJoin.enabled = true (ya está configurado en la Celda 1) y se hace con .join(df, lit(true), "cross").
El número de filas resultado es: número de farmacias × número de categorías únicas.
Resultado esperado: 8 farmacias × 3 categorías = 24 filas.

In [43]:
import org.apache.spark.sql.functions.col

// 1. Obtener las categorías únicas de productos
val categoriasUnicas = productos.select("categoria").distinct()

// 2. Preparar el DataFrame de farmacias con solo las columnas necesarias
val farmaciasBase = farmacias.select("farmaciaId", "nombre")

// 3. Generar el producto cartesiano
// Usamos el método crossJoin para combinar cada farmacia con cada categoría
val matrizAuditoria = farmaciasBase.crossJoin(categoriasUnicas)
  .orderBy(col("farmaciaId").asc, col("categoria").asc)

// 4. Mostrar resultado
println("=== Matriz de Plan de Auditoría Anual ===")
matrizAuditoria.show(24, truncate = false)

// Verificación de dimensiones
println(s"Total de combinaciones generadas: ${matrizAuditoria.count()} (8 farmacias x 3 categorías)")


=== Matriz de Plan de Auditoría Anual ===
+----------+-----------------+-----------+
|farmaciaId|nombre           |categoria  |
+----------+-----------------+-----------+
|1         |Farmacia Central |Analgésico |
|1         |Farmacia Central |Antibiótico|
|1         |Farmacia Central |Vitamina   |
|2         |Farmacia del Sol |Analgésico |
|2         |Farmacia del Sol |Antibiótico|
|2         |Farmacia del Sol |Vitamina   |
|3         |Farmacia Diagonal|Analgésico |
|3         |Farmacia Diagonal|Antibiótico|
|3         |Farmacia Diagonal|Vitamina   |
|4         |Farmacia Rambla  |Analgésico |
|4         |Farmacia Rambla  |Antibiótico|
|4         |Farmacia Rambla  |Vitamina   |
|5         |Farmacia Norte   |Analgésico |
|5         |Farmacia Norte   |Antibiótico|
|5         |Farmacia Norte   |Vitamina   |
|6         |Farmacia Ría     |Analgésico |
|6         |Farmacia Ría     |Antibiótico|
|6         |Farmacia Ría     |Vitamina   |
|7         |Farmacia Gran Vía|Analgésico |
|7         |

import org.apache.spark.sql.functions.col
categoriasUnicas: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [categoria: string]
farmaciasBase: org.apache.spark.sql.package.DataFrame = [farmaciaId: int, nombre: string]
matrizAuditoria: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [farmaciaId: int, nombre: string ... 1 more field]

Misión 9 — Análisis del plan de ejecución (reflexión)
Petición del negocio: "El equipo quiere documentar por qué algunas consultas son más rápidas que otras."

Lo que debes hacer:

Toma el join de la Misión 1 (ventas con productos) e implementa dos versiones: una sin broadcast y otra con broadcast(productos). Usa .explain() sobre cada una y responde en una celda de tipo Markdown las siguientes preguntas:

¿Qué tipo de join aparece en el plan sin broadcast?
¿Qué tipo de join aparece en el plan con broadcast?
¿Por qué tiene sentido aplicar broadcast sobre productos y no sobre ventas?
¿Qué columna del DataFrame ventas es la más adecuada para hacer el join con productos?

In [44]:
import org.apache.spark.sql.functions.{col, broadcast}

// Versión 1: Sin Broadcast (Join estándar)
val joinSinBroadcast = ventas.join(productos, "productoId")
println("=== PLAN SIN BROADCAST ===")
joinSinBroadcast.explain()

// Versión 2: Con Broadcast
val joinConBroadcast = ventas.join(broadcast(productos), "productoId")
println("\n=== PLAN CON BROADCAST ===")
joinConBroadcast.explain()


=== PLAN SIN BROADCAST ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [productoId#871, ventaId#869, farmaciaId#870, cantidad#872, importe#873, fecha#874, nombreProducto#893, categoria#894, requiereReceta#895]
   +- BroadcastHashJoin [productoId#871], [productoId#892], Inner, BuildRight, false
      :- LocalTableScan [ventaId#869, farmaciaId#870, productoId#871, cantidad#872, importe#873, fecha#874]
      +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, true]),false), [plan_id=5028]
         +- LocalTableScan [productoId#892, nombreProducto#893, categoria#894, requiereReceta#895]



=== PLAN CON BROADCAST ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [productoId#871, ventaId#869, farmaciaId#870, cantidad#872, importe#873, fecha#874, nombreProducto#893, categoria#894, requiereReceta#895]
   +- BroadcastHashJoin [productoId#871], [productoId#892], Inner, BuildRight, false
      :- LocalTableScan [ventaId#869, farmaciaI

import org.apache.spark.sql.functions.{col, broadcast}
joinSinBroadcast: org.apache.spark.sql.package.DataFrame = [productoId: string, ventaId: string ... 7 more fields]
joinConBroadcast: org.apache.spark.sql.package.DataFrame = [productoId: string, ventaId: string ... 7 more fields]

Misión 9: Reflexión sobre el Plan de Ejecución
1. ¿Qué tipo de join aparece en el plan sin broadcast?
En el plan físico aparecerá un SortMergeJoin (o un ShuffleHashJoin dependiendo de la configuración). Este tipo de join requiere un Shuffle, lo que significa que Spark debe mover y reordenar los datos de ambos DataFrames a través de la red para que las filas con el mismo productoId terminen en el mismo nodo.
2. ¿Qué tipo de join aparece en el plan con broadcast?
Aparece un BroadcastHashJoin. En este caso, Spark envía una copia completa del DataFrame pequeño (productos) a cada uno de los ejecutores del clúster. Esto elimina la necesidad de mover los datos del DataFrame grande (ventas), permitiendo que el cruce se realice localmente en cada nodo.
3. ¿Por qué tiene sentido aplicar broadcast sobre productos y no sobre ventas?
El broadcast tiene sentido sobre productos porque es una tabla pequeña (catálogo) que cabe fácilmente en la memoria de los ejecutores. Hacer broadcast de ventas sería ineficiente y peligroso (podría causar errores de Out of Memory), ya que las tablas de hechos o transacciones suelen ser masivas y superan la capacidad de memoria de un solo nodo.
4. ¿Qué columna del DataFrame ventas es la más adecuada para hacer el join con productos?
La columna más adecuada es productoId. Es la "Foreign Key" (clave foránea) que vincula cada transacción de venta con los detalles técnicos, la categoría y el nombre del catálogo de productos.